_ExplainDB — Copyright (C) 2026 Prof. Dr. Jens Dittrich, Saarland University._  
_Licensed under the GNU Affero General Public License v3 (AGPL-3.0). See the `LICENSE` file._


**How would we translate a query like this?**
``` sql
SELECT * 
FROM books
ORDER BY title
LIMIT 10
```




**Create some fake data**


In [1]:
from faker import Faker

fake = Faker()
fake.seed_instance(42)

from dataclasses import dataclass
import heapq
from functools import total_ordering

number_of_tuples = 40


@total_ordering
@dataclass(eq=False)
class Book:
    author: str
    title: str

    def __eq__(self, other):
        return self.title == other.title

    def __lt__(self, other):
        return self.title > other.title


# create some fake data mimicking tuples in a relation:
data = [Book(fake.name(), fake.sentence()) for f in range(number_of_tuples)]
data[:10]

[Book(author='Nicholas Parker', title='None join yeah.'),
 Book(author='Debra Miller', title='Company worker natural himself add need.'),
 Book(author='Kimberly Garza', title='Along other along news toward.'),
 Book(author='Christopher Edwards', title='Worry top administration provide about reach.'),
 Book(author='Kim Fitzpatrick', title='Kitchen wife right successful law.'),
 Book(author='Amanda Roberson', title='Third ask few night.'),
 Book(author='Haley Turner', title='Hair mission cultural wish shoulder.'),
 Book(author='Lisa Smith', title='Marriage chair person guy deep future senior.'),
 Book(author='Cynthia Fernandez', title='Play plan number entire push child.'),
 Book(author='Jennifer Reese', title='Cultural beautiful subject relate here side happen.')]

**Algorithm 1: Sort and Cut**

In [2]:
def sort_and_cut(data, k=10):
    # make a deep copy to start with the unchanged data:
    books = data.copy()

    # handle the case that k is larger than the data:
    k = min(len(data), k)

    # standard sort:
    books.sort(reverse=True)

    # cut list to just show the first k elements:
    return books[:k]

    # so we throw everything away except the first ten elements, that is smart, not...


sort_and_cut(data)

[Book(author='Garrett Myers', title='All three money her.'),
 Book(author='Kimberly Garza', title='Along other along news toward.'),
 Book(author='Bill Reyes', title='Baby exist next current.'),
 Book(author='Charles Rivera', title='Bill you arrive quality environment significant hand.'),
 Book(author='Mrs. Lisa Bird', title='Card important forward sport yard care through baby.'),
 Book(author='Pamela Davis', title='Clearly indeed our discuss among prevent somebody vote.'),
 Book(author='Debra Miller', title='Company worker natural himself add need.'),
 Book(author='Jennifer Reese', title='Cultural beautiful subject relate here side happen.'),
 Book(author='Deborah Powell', title='Drug quality visit modern ability side.'),
 Book(author='Theresa Callahan', title='Easy field compare significant.')]

Is that all we can do?

What if the table is large?

We would have to first fully sort the table and than pick the top-10 elements. That is inefficient.

**Algorithm 2: Queue-based Scan**

In [3]:
def queue_based_scan(data, k=10):
    """Scan input stream keeping the currently seen smallest k elements in a max-heap.
    This is an example of partial sorting. Note that the heap-based implementation only
    requires O(k) memory and has a time complexity of O(n log k) where n is the number of
    elements in the input stream. This is much more efficient than sorting the whole array
    first and then selecting the top-k elements (as done above).
    """

    # make a deep copy to start with the unchanged data:
    books = data.copy()

    # handle the case that k is larger than the data:
    k = min(len(data), k)

    # just copy the first 10 elements:
    result = books[:k].copy()

    # create a max heap on these elements (in linear time):
    heapq.heapify(result)

    # for all remaining elements from the dataset:
    for book in books[k:]:
        # is the current element smaller than the top-element? (watch out for ties)
        if book.title < result[0].title:
            # remove the current max element from the heap and replace it with book:
            # also establish heap-properties afterwards
            heapq.heappushpop(result, book)

    # from the heap create a sorted list
    # note: just popping each element and then reversing would do the same
    result.sort(reverse=True)

    return result


queue_based_scan(data)

[Book(author='Garrett Myers', title='All three money her.'),
 Book(author='Kimberly Garza', title='Along other along news toward.'),
 Book(author='Bill Reyes', title='Baby exist next current.'),
 Book(author='Charles Rivera', title='Bill you arrive quality environment significant hand.'),
 Book(author='Mrs. Lisa Bird', title='Card important forward sport yard care through baby.'),
 Book(author='Pamela Davis', title='Clearly indeed our discuss among prevent somebody vote.'),
 Book(author='Debra Miller', title='Company worker natural himself add need.'),
 Book(author='Jennifer Reese', title='Cultural beautiful subject relate here side happen.'),
 Book(author='Deborah Powell', title='Drug quality visit modern ability side.'),
 Book(author='Theresa Callahan', title='Easy field compare significant.')]

**Unit Tests**

In [4]:
import unittest


class MyTest(unittest.TestCase):

    # the following function will be called before every test-function:

    def create_data(self, number_of_tuples):
        self.data = [
            Book(fake.name(), fake.sentence()) for f in range(number_of_tuples)
        ]

    def test_topk_results_end_to_end(self):
        for i in range(10):
            self.create_data(1000)
            res1 = sort_and_cut(self.data)
            res2 = queue_based_scan(self.data)
            self.assertListEqual(res1, res2)


# Run the unit test without shutting down the jupyter kernel
# notice that this call will look for all test classes, i.e.
# all classes inheriting from unittest.TestCase
# each of these functions will be executed once
# unittest.main(argv=['ignored', '-v'], verbosity=2, exit=False)

# only execute a specific Test-class:
unittest.main(argv=["ignored", "-v"], defaultTest="MyTest", verbosity=2, exit=False)

test_topk_results_end_to_end (__main__.MyTest.test_topk_results_end_to_end) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.527s

OK
